# Code 10.4.2: TD(0) prediction

Updates after every step toward the one-step TD target, and measures the RMS error of both methods against the exact values from Code 10.2.2 (the 100-run averages plotted in Figure 10.9 repeat this with seeds 1000 to 1099).

Earlier imports and definitions this listing relies on are included below so the notebook runs on its own.


In [ ]:
import sys
from pathlib import Path

def _add_gridworld_path():
    """The chapter figure scripts ship the gridworld module this listing imports."""
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        candidate = base / "chapters/10-reinforcement-learning-basics/figures/src/gridworld.py"
        if candidate.is_file():
            folder = str(candidate.parent)
            if folder not in sys.path:
                sys.path.insert(0, folder)
            return
    raise FileNotFoundError(
        "Could not find chapters/10-reinforcement-learning-basics/figures/src/gridworld.py. "
        "Run this notebook from the repository (or a subdirectory of it)."
    )

_add_gridworld_path()


### Setup from Code 10.4.1

Included so the next cell can reuse its definitions and variables.


In [ ]:
import numpy as np
from gridworld import GridWorld

env = GridWorld()
GAMMA = 0.9
nonterminal = np.flatnonzero(~env.terminal)

def generate_episode(env, rng):
    """One episode of the uniform random policy; returns states and rewards."""
    s = rng.choice(nonterminal)
    states, rewards = [], []
    done = False
    while not done:
        a = rng.integers(env.n_actions)
        s2, r, done = env.step(s, a)
        states.append(s)
        rewards.append(r)
        s = s2
    return states, rewards

def mc_prediction(env, n_episodes, rng, alpha=None, gamma=GAMMA):
    """Every-visit Monte Carlo: move V(s) toward each observed return G_t."""
    V, N = np.zeros(env.n_states), np.zeros(env.n_states)
    history = []
    for _ in range(n_episodes):
        states, rewards = generate_episode(env, rng)
        G = 0.0
        for s, r in zip(reversed(states), reversed(rewards)):   # walk backward
            G = r + gamma * G                                   # return from s
            N[s] += 1
            step = 1.0 / N[s] if alpha is None else alpha
            V[s] += step * (G - V[s])
        history.append(V.copy())
    return np.array(history)


In [ ]:
from gridworld import policy_evaluation, random_policy

def td0_prediction(env, n_episodes, rng, alpha, gamma=GAMMA):
    """TD(0): after every step, move V(s) toward r + gamma * V(s')."""
    V = np.zeros(env.n_states)
    history = []
    for _ in range(n_episodes):
        s = rng.choice(nonterminal)
        done = False
        while not done:
            a = rng.integers(env.n_actions)
            s2, r, done = env.step(s, a)
            target = r + (0.0 if done else gamma * V[s2])
            V[s] += alpha * (target - V[s])
            s = s2
        history.append(V.copy())
    return np.array(history)

V_true = policy_evaluation(env, random_policy(env), GAMMA)
def rms(history):
    return np.sqrt(((history[:, nonterminal] - V_true[nonterminal]) ** 2).mean(axis=1))

for alpha in (0.02, 0.05):
    mc = np.mean([rms(mc_prediction(env, 300, np.random.default_rng(1000 + i), alpha))
                  for i in range(100)], axis=0)
    td = np.mean([rms(td0_prediction(env, 300, np.random.default_rng(1000 + i), alpha))
                  for i in range(100)], axis=0)
    print(f"alpha={alpha}: RMS error after 300 episodes  MC {mc[-1]:.3f}  TD(0) {td[-1]:.3f}")


**Expected output**

Output:

```text
alpha=0.02: RMS error after 300 episodes  MC 0.063  TD(0) 0.036
alpha=0.05: RMS error after 300 episodes  MC 0.097  TD(0) 0.045
```
